In [7]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [8]:
# Artifact paths
artifacts_dir = Path("data/artifacts")
features_dir = artifacts_dir / "features"

features_dir.mkdir(parents=True, exist_ok=True)

print("Features directory:", features_dir)

Features directory: data\artifacts\features


In [9]:
train = pd.read_parquet(
    artifacts_dir / "03_train1.parquet"
)

validation = pd.read_parquet(
    artifacts_dir / "03_validation1.parquet"
)

test = pd.read_parquet(
    artifacts_dir / "03_test1.parquet"
)

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67529, 42)
Validation shape: (14470, 42)
Test shape: (14471, 42)


In [10]:
def create_time_features(df):
    df = df.copy()

    # Make sure datetime columns are datetime
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"],
        errors="coerce"
    )

    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"],
        errors="coerce"
    )

    # Purchase time features
    df["purchase_month"] = (
        df["order_purchase_timestamp"].dt.month
    )

    df["purchase_weekday"] = (
        df["order_purchase_timestamp"].dt.weekday
    )

    df["purchase_hour"] = (
        df["order_purchase_timestamp"].dt.hour
    )

    # Estimated delivery duration
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)

    return df

In [11]:
train_fe = create_time_features(train)
validation_fe = create_time_features(validation)
test_fe = create_time_features(test)

print("Time features created successfully.")

Time features created successfully.


In [12]:
numeric_features = [
    "seller_customer_distance_km",
    "total_freight",
    "total_price",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_delivery_days"
]

categorical_features = [
    "customer_state"
]

target = "is_delayed"

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTarget:")
print(target)

Numeric features:
['seller_customer_distance_km', 'total_freight', 'total_price', 'num_items', 'num_unique_products', 'num_unique_sellers', 'total_product_weight_g', 'total_product_volume_cm3', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'estimated_delivery_days']

Categorical features:
['customer_state']

Target:
is_delayed


In [13]:
X_train = train_fe[
    numeric_features + categorical_features
].copy()

X_validation = validation_fe[
    numeric_features + categorical_features
].copy()

X_test = test_fe[
    numeric_features + categorical_features
].copy()

y_train = train_fe[target].copy()
y_validation = validation_fe[target].copy()
y_test = test_fe[target].copy()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

X_train: (67529, 13)
X_validation: (14470, 13)
X_test: (14471, 13)


In [14]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [15]:
X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(
    X_validation
)

X_test_processed = preprocessor.transform(
    X_test
)

print("Train processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_validation_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Train processed shape: (67529, 39)
Validation processed shape: (14470, 39)
Test processed shape: (14471, 39)


In [16]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))

print("\nFeature names:")
for i, feature in enumerate(feature_names, start=1):
    print(i, feature)

Number of features: 39

Feature names:
1 numeric__seller_customer_distance_km
2 numeric__total_freight
3 numeric__total_price
4 numeric__num_items
5 numeric__num_unique_products
6 numeric__num_unique_sellers
7 numeric__total_product_weight_g
8 numeric__total_product_volume_cm3
9 numeric__purchase_month
10 numeric__purchase_weekday
11 numeric__purchase_hour
12 numeric__estimated_delivery_days
13 categorical__customer_state_AC
14 categorical__customer_state_AL
15 categorical__customer_state_AM
16 categorical__customer_state_AP
17 categorical__customer_state_BA
18 categorical__customer_state_CE
19 categorical__customer_state_DF
20 categorical__customer_state_ES
21 categorical__customer_state_GO
22 categorical__customer_state_MA
23 categorical__customer_state_MG
24 categorical__customer_state_MS
25 categorical__customer_state_MT
26 categorical__customer_state_PA
27 categorical__customer_state_PB
28 categorical__customer_state_PE
29 categorical__customer_state_PI
30 categorical__customer_st

In [17]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

print("Train:", X_train_processed.shape)
print("Validation:", X_validation_processed.shape)
print("Test:", X_test_processed.shape)

Train: (67529, 39)
Validation: (14470, 39)
Test: (14471, 39)


In [18]:
print("Missing values in processed data:")

print(
    "Train:",
    X_train_processed.isna().sum().sum()
)

print(
    "Validation:",
    X_validation_processed.isna().sum().sum()
)

print(
    "Test:",
    X_test_processed.isna().sum().sum()
)

Missing values in processed data:
Train: 0
Validation: 0
Test: 0


In [20]:
preprocessor_path = (
    features_dir / "05_preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    preprocessor_path
)

print(
    "Preprocessor saved to:",
    preprocessor_path
)

Preprocessor saved to: data\artifacts\features\05_preprocessor.joblib


In [21]:
X_train_processed.to_parquet(
    features_dir / "05_X_train.parquet",
    index=False
)

X_validation_processed.to_parquet(
    features_dir / "05_X_validation.parquet",
    index=False
)

X_test_processed.to_parquet(
    features_dir / "05_X_test.parquet",
    index=False
)

print("Processed feature tables saved.")

Processed feature tables saved.


In [22]:
y_train.to_frame().to_parquet(
    features_dir / "05_y_train.parquet",
    index=False
)

y_validation.to_frame().to_parquet(
    features_dir / "05_y_validation.parquet",
    index=False
)

y_test.to_frame().to_parquet(
    features_dir / "05_y_test.parquet",
    index=False
)

print("Target files saved.")

Target files saved.


In [23]:
feature_list = pd.DataFrame({
    "feature_name": feature_names
})

feature_list.to_csv(
    features_dir / "05_feature_list.csv",
    index=False
)

print("Feature list saved.")

Feature list saved.


In [24]:
print("=" * 50)
print("FINAL FEATURE ENGINEERING CHECK")
print("=" * 50)

print("Train X shape:", X_train_processed.shape)
print("Validation X shape:", X_validation_processed.shape)
print("Test X shape:", X_test_processed.shape)

print("\nTrain y shape:", y_train.shape)
print("Validation y shape:", y_validation.shape)
print("Test y shape:", y_test.shape)

print("\nNumber of features:", len(feature_names))

print(
    "\nMissing values - Train:",
    X_train_processed.isna().sum().sum()
)

print(
    "Missing values - Validation:",
    X_validation_processed.isna().sum().sum()
)

print(
    "Missing values - Test:",
    X_test_processed.isna().sum().sum()
)

print("\nArtifacts:")
for file in sorted(features_dir.iterdir()):
    print("-", file.name)

FINAL FEATURE ENGINEERING CHECK
Train X shape: (67529, 39)
Validation X shape: (14470, 39)
Test X shape: (14471, 39)

Train y shape: (67529,)
Validation y shape: (14470,)
Test y shape: (14471,)

Number of features: 39

Missing values - Train: 0
Missing values - Validation: 0
Missing values - Test: 0

Artifacts:
- 05_feature_list.csv
- 05_preprocessor.joblib
- 05_X_test.parquet
- 05_X_train.parquet
- 05_X_validation.parquet
- 05_y_test.parquet
- 05_y_train.parquet
- 05_y_validation.parquet
